# Capítulo 2 — Entendendo os modelos de base

**Livro:** começa na página **123** do PDF.

**Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Modelos multilíngues · PDF p. 126

#### Exercício: Quanto custa falar português?

Pegue um parágrafo (ex.: um trecho do próprio livro) e peça ao modelo para traduzi-lo para inglês, espanhol e japonês. Compare quantos tokens o texto tem em cada idioma. Qual idioma é mais caro de processar?

In [ ]:
texto_pt = ("A engenharia de IA se concentra em adaptar modelos já treinados a aplicações reais, "
            "usando prompts, recuperação de contexto e ajuste fino quando necessário.")
tamanhos = {"português": contar_tokens(texto_pt)}
for idioma in ["inglês", "espanhol", "japonês"]:
    traducao = perguntar(f"Traduza para {idioma}. Responda só com a tradução:\n\n{texto_pt}", temperature=0)
    tamanhos[idioma] = contar_tokens(traducao)
tamanhos

### Fundamentos de Amostragem · PDF p. 190

#### Exercício: Ver as probabilidades do próximo token

Peça ao modelo para completar a frase *"A capital do Brasil é"* com **1 token**, pedindo `logprobs=True, top_logprobs=5`. Mostre os 5 tokens candidatos e transforme o logprob em probabilidade (`math.exp`).

In [ ]:
import math
r = conversar([{"role": "user", "content": "Complete com uma palavra: A capital do Brasil é"}],
              max_tokens=1, logprobs=True, top_logprobs=5, temperature=0)
candidatos = r.choices[0].logprobs.content[0].top_logprobs
for c in candidatos:
    print(f"{c.token!r:>12}  probabilidade = {math.exp(c.logprob):.3f}")

### Estratégias de amostragem · PDF p. 194

#### Exercício: Temperatura: criatividade × repetição

Peça 5 vezes um *nome para uma farmácia de bairro* com `temperature=0` e 5 vezes com `temperature=1.5`. Quantos nomes diferentes saíram em cada caso? Relacione com a explicação de temperatura das p. 194–197.

In [ ]:
def nomes(temperatura, n=5):
    return [perguntar("Sugira um nome para uma farmácia de bairro. Responda só o nome.",
                      temperature=temperatura, max_tokens=15).strip() for _ in range(n)]

frios, quentes = nomes(0), nomes(1.5)
print("temperatura 0  :", frios, "-> diferentes:", len(set(frios)))
print("temperatura 1.5:", quentes, "-> diferentes:", len(set(quentes)))

### Tempo de teste Calcular · PDF p. 202

#### Exercício: Gastar mais na hora da resposta: votação da maioria

Faça uma pergunta de raciocínio (ex.: *"Uma caneta e um caderno custam R$ 11; o caderno custa R$ 10 a mais que a caneta. Quanto custa a caneta?"*). Gere 7 respostas com temperatura 1, pedindo **só o número no final**, e escolha a resposta mais frequente (votação). Compare com uma única resposta.

In [ ]:
from collections import Counter
import re

pergunta = ("Uma caneta e um caderno custam R$ 11 juntos. O caderno custa R$ 10 a mais que a caneta. "
            "Quanto custa a caneta? Pense passo a passo e termine com 'RESPOSTA: <valor>'.")
respostas = []
for _ in range(7):
    texto = perguntar(pergunta, temperature=1)
    achado = re.search(r"RESPOSTA:\s*R?\$?\s*([\d.,]+)", texto)
    respostas.append(achado.group(1) if achado else "?")
print("Respostas:", respostas)
print("Mais votada:", Counter(respostas).most_common(1)[0])  # o certo é 0,50

### Saídas estruturadas · PDF p. 207

#### Exercício: Pedir JSON e validar

Peça ao modelo para extrair de um texto de pedido: **produto, quantidade e data de entrega**, em JSON (`response_format={'type': 'json_object'}`). Valide a resposta com um modelo `pydantic`.

In [ ]:
import json
from pydantic import BaseModel

class Pedido(BaseModel):
    produto: str
    quantidade: int
    data_entrega: str

texto = "Precisamos de 300 caixas de dipirona 500 mg entregues até 15/12/2026, por favor."
resposta = perguntar(
    "Extraia produto, quantidade (inteiro) e data_entrega (AAAA-MM-DD) do texto. "
    "Responda só com um objeto JSON com essas chaves.\n\nTexto: " + texto,
    temperature=0, response_format={"type": "json_object"})
pedido = Pedido(**json.loads(resposta))
pedido

### A natureza probabilística da IA · PDF p. 217

#### Exercício: A mesma pergunta, respostas diferentes

Faça 10 vezes a mesma pergunta de opinião (ex.: *"Qual a melhor linguagem para análise de dados? Responda em uma palavra."*) com temperatura 1. Conte as respostas. O que isso significa para quem coloca IA em produção (p. 217–226)?

In [ ]:
from collections import Counter
respostas = [perguntar("Qual a melhor linguagem para análise de dados? Responda em uma palavra.",
                       temperature=1, max_tokens=5).strip(" .").lower() for _ in range(10)]
Counter(respostas)
# Inconsistência: o mesmo pedido pode dar respostas diferentes. Em produção, use temperatura baixa,
# saídas estruturadas e avaliação contínua.

## Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou